In [59]:
import pandas as pd
import matplotlib.pyplot as plt

from pyvis.network import Network
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output
import IPython

import json

In [ ]:
# 1. Charger et préparer les données
with open('../data/data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)

col_etab = 'etablissement_norm' if 'etablissement_norm' in df.columns else 'etablissement'
df_exploded = df.explode('directeurs').dropna(subset=['directeurs', col_etab])

# 2. Créer la liste complète de tous les liens (Etablissement <-> Directeur)
edges_df = df_exploded[[col_etab, 'directeurs']].drop_duplicates()
edges_list = edges_df.rename(columns={col_etab: 'from', 'directeurs': 'to'}).to_dict('records')

# On convertit cette liste en format JSON pour l'injecter dans notre fichier HTML
edges_json = json.dumps(edges_list)

# 3. Création du code HTML et JavaScript sur mesure
html_content = f"""
<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <title>Réseau Interactif M@ïeutIC</title>
    <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background-color: #f4f7f6; }}
        h2 {{ color: #2c3e50; }}
        #mynetwork {{ width: 100%; height: 750px; border: 1px solid #ccd1d9; background-color: #ffffff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
        .controls {{ padding: 20px; background: #ffffff; border: 1px solid #ccd1d9; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
        select {{ padding: 10px; margin-right: 20px; font-size: 15px; min-width: 250px; border: 1px solid #bdc3c7; border-radius: 4px; }}
        label {{ font-weight: bold; margin-right: 10px; color: #34495e; }}
        #message {{ color: #e67e22; font-weight: bold; margin-bottom: 15px; font-size: 16px; display: none; }}
    </style>
</head>
<body>
    <h2>Explorateur de Réseau : Établissements & Directeurs de Thèse</h2>
    
    <div class="controls">
        <label for="etabSelect">Établissement :</label>
        <select id="etabSelect"><option value="Tous">Sélectionnez un établissement...</option></select>

        <label for="dirSelect">Directeur / Directrice :</label>
        <select id="dirSelect"><option value="Tous">Tous</option></select>
    </div>
    
    <div id="message">Sélectionnez un établissement pour afficher son réseau d'encadrement (la base de données complète est trop vaste pour être affichée d'un seul coup).</div>
    
    <div id="mynetwork"></div>

    <script type="text/javascript">
        // 1. Récupération des données depuis Python
        const allEdges = {edges_json};
        
        const etabSelect = document.getElementById('etabSelect');
        const dirSelect = document.getElementById('dirSelect');
        const messageDiv = document.getElementById('message');
        const container = document.getElementById('mynetwork');
        
        let network = null;

        // 2. Initialisation des listes
        // On récupère tous les établissements uniques et on les trie
        const etabs = [...new Set(allEdges.map(e => e.from))].sort();
        etabs.forEach(e => etabSelect.add(new Option(e, e)));

        function updateDirSelect() {{
            const selectedEtab = etabSelect.value;
            dirSelect.innerHTML = '<option value="Tous">Tous</option>';
            
            let validDirs = new Set();
            allEdges.forEach(e => {{
                if (selectedEtab === 'Tous' || e.from === selectedEtab) {{
                    validDirs.add(e.to);
                }}
            }});
            
            [...validDirs].sort().forEach(d => dirSelect.add(new Option(d, d)));
        }}

        // 3. Fonction pour générer le graphe
        function drawGraph() {{
            const selectedEtab = etabSelect.value;
            const selectedDir = dirSelect.value;

            // Sécurité : on empêche l'affichage global pour ne pas faire crasher le navigateur
            if (selectedEtab === 'Tous' && selectedDir === 'Tous') {{
                messageDiv.style.display = 'block';
                if (network) network.destroy();
                return;
            }} else {{
                messageDiv.style.display = 'none';
            }}

            let filteredEdges = [];
            let nodeIds = new Set();

            // Filtrage des liens selon les listes déroulantes
            allEdges.forEach(e => {{
                const matchEtab = (selectedEtab === 'Tous' || e.from === selectedEtab);
                const matchDir = (selectedDir === 'Tous' || e.to === selectedDir);

                if (matchEtab && matchDir) {{
                    filteredEdges.push({{from: e.from, to: e.to}});
                    nodeIds.add(e.from);
                    nodeIds.add(e.to);
                }}
            }});

            // Création des nœuds visuels
            let nodes = [];
            nodeIds.forEach(id => {{
                if (etabs.includes(id)) {{
                    // Style pour les établissements (Bleu, forme de boîte)
                    nodes.push({{id: id, label: id, color: '#4C72B0', shape: 'box', font: {{color: 'white'}}}});
                }} else {{
                    // Style pour les directeurs (Vert, forme de point)
                    nodes.push({{id: id, label: id, color: '#55A868', shape: 'dot'}});
                }}
            }});

            const data = {{
                nodes: new vis.DataSet(nodes),
                edges: new vis.DataSet(filteredEdges)
            }};

            const options = {{
                physics: {{
                    repulsion: {{ nodeDistance: 150 }},
                    solver: 'repulsion'
                }},
                edges: {{ color: '#888888', width: 1.5 }}
            }};

            // Redessiner le réseau
            if (network) network.destroy();
            network = new vis.Network(container, data, options);
        }}

        // 4. Écouteurs d'événements (ce qui se passe quand on clique)
        etabSelect.addEventListener('change', () => {{
            updateDirSelect();
            drawGraph();
        }});

        dirSelect.addEventListener('change', drawGraph);

        // Lancement initial
        updateDirSelect();
        drawGraph();
    </script>
</body>
</html>
"""

# 4. Enregistrement du fichier HTML autonome
with open('reseau_interactif_complet.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✅ Fichier 'reseau_interactif_complet.html' généré avec succès ! Tu peux l'ouvrir dans ton navigateur web.")

✅ Fichier 'reseau_interactif_complet.html' généré avec succès ! Tu peux l'ouvrir dans ton navigateur web.
